# 10 · Las 10 tiendas de M5

¿El efecto de corregir la demanda censurada se repite fuera de CA_1? `src/multitienda.py` empaqueta el pipeline completo (datos → detector → LightGBM → TSB con ventas y con demanda corregida → simulación (R, S) con calibración clase × quintil de volumen) y se ejecuta en las 10 tiendas (California, Texas y Wisconsin).

- **Control:** CA_1 debe reproducir exactamente `06_ablacion_censura`.
- AutoETS se omite (~20 min por tienda); TSB representa a los métodos clásicos.
- Cada tienda tarda ~3 min; los resultados se guardan en `reports/multitienda/` y se reutilizan si ya existen.

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW, REP, FIG_DIR = ROOT / "data" / "raw", ROOT / "reports", ROOT / "reports" / "figures"
OUT = REP / "multitienda"
OUT.mkdir(exist_ok=True)
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))
from multitienda import preparar, correr_tienda

TIENDAS = ["CA_1", "CA_2", "CA_3", "CA_4", "TX_1", "TX_2", "TX_3", "WI_1", "WI_2", "WI_3"]

for t in TIENDAS:
    if (OUT / f"{t}.csv").exists():
        continue
    t0 = time.time()
    r = correr_tienda(preparar(RAW, t))
    r["tabla"].assign(tienda=t).to_csv(OUT / f"{t}.csv", index=False)
    pd.Series({"productos": r["productos"], "agotado_pct": r["agotado_pct"], "imputado_pct": r["imputado_pct"],
               **{f"sesgo_{m}": v for m, v in r["sesgo_sim_pct"].items()}}).to_json(OUT / f"{t}_meta.json")
    print(f"{t}: {r['productos']:,} productos | agotado {r['agotado_pct']:.2f}% | {(time.time() - t0) / 60:.1f} min")

CA_1: 2,467 productos | agotado 2.69% | 1.8 min


CA_2: 2,279 productos | agotado 2.29% | 1.6 min


CA_3: 2,755 productos | agotado 2.86% | 1.7 min


CA_4: 2,393 productos | agotado 1.65% | 2.0 min


TX_1: 2,493 productos | agotado 1.75% | 2.0 min


TX_2: 2,516 productos | agotado 2.28% | 1.8 min


TX_3: 2,442 productos | agotado 1.97% | 1.7 min


WI_1: 2,516 productos | agotado 2.25% | 1.4 min


WI_2: 2,426 productos | agotado 2.35% | 1.4 min


WI_3: 2,409 productos | agotado 2.18% | 1.4 min


In [2]:
# Control: CA_1 reproduce 06_ablacion_censura
ca1 = pd.read_csv(OUT / "CA_1.csv")
ref = pd.read_csv(REP / "ablacion_censura_CA_1.csv")
chk = ref.merge(ca1, on=["modelo", "alpha"], suffixes=("_06", "_10"))
assert len(chk) == 4 * 7 and np.allclose(chk["costo_total_06"], chk["costo_total_10"], rtol=1e-6), "CA_1 no reproduce 06"
print("Control OK: CA_1 reproduce 06_ablacion_censura")

Control OK: CA_1 reproduce 06_ablacion_censura


In [3]:
def inv_en(t, modelo, fr=0.95):
    c = t[t["modelo"] == modelo].sort_values("fill_rate")
    return np.interp(fr, c["fill_rate"], c["inv_valor_prom"]) if c["fill_rate"].min() <= fr <= c["fill_rate"].max() else np.nan

filas = []
for t in TIENDAS:
    tab = pd.read_csv(OUT / f"{t}.csv")
    meta = pd.read_json(OUT / f"{t}_meta.json", typ="series")
    opt = tab.loc[tab.groupby("modelo")["costo_total"].idxmin()].set_index("modelo")["costo_total"]
    inv = {m: inv_en(tab, m) for m in ["LGB", "TSB", "TSB_imp"]}
    filas.append({
        "tienda": t, "productos": int(meta["productos"]), "agotado_pct": meta["agotado_pct"],
        "sesgo_TSB_pct": meta["sesgo_TSB"], "sesgo_TSB_imp_pct": meta["sesgo_TSB_imp"], "sesgo_LGB_pct": meta["sesgo_LGB"],
        "censura_costo_pct": (opt["TSB_imp"] / opt["TSB"] - 1) * 100,
        "censura_inv95_pct": (inv["TSB_imp"] / inv["TSB"] - 1) * 100,
        "modelo_costo_pct": (opt["LGB"] / opt["TSB_imp"] - 1) * 100,
        "modelo_inv95_pct": (inv["LGB"] / inv["TSB_imp"] - 1) * 100,
        "lgb_vs_tsb_costo_pct": (opt["LGB"] / opt["TSB"] - 1) * 100,
    })
res = pd.DataFrame(filas)
res.to_csv(REP / "multitienda_resumen.csv", index=False)
display(res.round(2))
print(res[["censura_costo_pct", "censura_inv95_pct", "modelo_costo_pct", "modelo_inv95_pct"]].describe().round(1))

,tienda,productos,agotado_pct,sesgo_TSB_pct,sesgo_TSB_imp_pct,sesgo_LGB_pct,censura_costo_pct,censura_inv95_pct,modelo_costo_pct,modelo_inv95_pct,lgb_vs_tsb_costo_pct
0,CA_1,2467,2.69,-2.49,2.55,0.39,-27.59,-23.77,2.06,-1.76,-26.10
1,CA_2,2279,2.29,-3.07,1.40,-5.30,-22.70,-18.58,8.54,7.93,-16.10
2,CA_3,2755,2.86,-1.22,2.03,-0.48,-12.23,-9.49,6.97,0.24,-6.11
3,CA_4,2393,1.65,-2.33,1.61,-0.33,-16.62,-17.43,-2.97,-9.31,-19.10
4,TX_1,2493,1.75,-3.43,1.14,1.56,-19.66,-21.17,5.93,3.12,-14.90
5,TX_2,2516,2.28,-2.98,1.70,-2.59,-21.71,-23.45,7.74,2.78,-15.65
6,TX_3,2442,1.97,-3.63,1.50,-1.61,-20.69,-21.24,6.56,-0.29,-15.49
7,WI_1,2516,2.25,-4.03,0.32,-0.51,-22.21,-17.92,-1.21,-8.44,-23.16
8,WI_2,2426,2.35,-6.18,-1.00,-5.51,-19.63,-24.90,2.33,-7.49,-17.76
9,WI_3,2409,2.18,-3.93,0.04,-4.13,-19.01,-19.28,3.12,-2.52,-16.49


       censura_costo_pct  censura_inv95_pct  modelo_costo_pct  \
count               10.0               10.0              10.0   
mean               -20.2              -19.7               3.9   
std                  4.0                4.4               3.9   
min                -27.6              -24.9              -3.0   
25%                -22.1              -22.9               2.1   
50%                -20.2              -20.2               4.5   
75%                -19.2              -18.1               6.9   
max                -12.2               -9.5               8.5   

       modelo_inv95_pct  
count              10.0  
mean               -1.6  
std                 5.6  
min                -9.3  
25%                -6.2  
50%                -1.0  
75%                 2.1  
max                 7.9  


In [4]:
graf = res.melt(id_vars="tienda", value_vars=["censura_costo_pct", "modelo_costo_pct"],
                var_name="efecto", value_name="cambio")
graf["efecto"] = graf["efecto"].map({"censura_costo_pct": "Corregir la censura (TSB corregido vs. TSB)",
                                     "modelo_costo_pct": "Cambiar de modelo (LightGBM vs. TSB corregido)"})
fig = px.strip(graf, x="cambio", y="efecto", color="efecto", hover_data=["tienda"],
               labels={"cambio": "Cambio en costo total óptimo (%)", "efecto": ""},
               title="Efecto de la censura vs. efecto del modelo — 10 tiendas de M5")
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_traces(marker_size=11)
fig.update_layout(width=1000, height=420, showlegend=False)
fig.write_html(FIG_DIR / "11_multitienda.html", include_plotlyjs="cdn")
try:
    fig.write_image(FIG_DIR / "11_multitienda.png", scale=2)
except Exception as e:
    print(f"PNG no generado ({e})")
fig.show()

## Conclusiones

- **El efecto de la censura se repite en las 10 tiendas.** Entrenar TSB con demanda corregida reduce el costo total óptimo entre 12% y 28% (promedio −20%) y el inventario a fill rate de 95% entre 9% y 25% (promedio −20%). CA_1 es la tienda con el efecto más grande; el valor típico es ~20%.
- **El modelo sigue sin ser la palanca.** LightGBM entrenado una vez cuesta en promedio 4% *más* que TSB corregido (de −3% a +8.5% según la tienda), con inventario similar (−1.6% en promedio, de −9% a +8%). En CA_2 y WI_2 su sesgo llega a −5%: un modelo que no se reentrena se degrada en 9 meses (ver `09_reentrenamiento`).
- La tasa de agotados detectados es estable entre tiendas (1.6–2.9% de los días-producto) y en todas los métodos clásicos con ventas subestiman la demanda (sesgo de −1% a −6%).